# Part 1 - Data Cleaning with Python and pandas
**Project: CareLoop Clinics - No-Show Guard**

CareLoop runs 8 clinics. About 1 in 5 booked patients does not turn up (a "no-show").
In this notebook we take the raw appointment file, check it, clean it, and save a tidy
version that Part 2 (SQL), Part 3 (machine learning) and Part 4 (Gemini) will use.

| Step | What we do |
|---|---|
| 1.1 | Read the CSV with plain Python (`csv.DictReader`) and reject rows that fail a type check |
| 1.2 | Load with pandas, remove duplicates, clean text, parse dates |
| 1.3 | Fix invalid ages, fill missing values, cap outliers |
| 1.4 | Create new columns and do NumPy calculations |
| 1.5 | Call the Open-Meteo API for each clinic |
| 1.6 | Export the clean files and the SQL load script |

## Setup
The first cell installs the libraries (needed on Google Colab; on a laptop it finishes quickly
when they are already installed). The second cell finds the project folder so that the same
relative paths (`data/...`, `outputs/...`) work on a laptop and on Colab.

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

# This notebook lives in notebooks/. The data lives one level up, in data/.
# BASE is the project's top folder, wherever the notebook is started from.
BASE = Path(".") if Path("data").exists() else Path("..")
DATA = BASE / "data"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)   # create outputs/ if it does not exist

print("Data files found:", sorted(p.name for p in DATA.glob("*.csv")))

Data files found: ['appointments.csv', 'clinics.csv', 'followup_notes.csv']


## Step 1.1 - Read the raw file with `csv.DictReader` and reject bad rows

**What:** read `appointments.csv` using only Python's built-in `csv` module (no pandas yet).
Every value in a CSV file is text. `parse_row` tries to turn the number columns into real
numbers. If a conversion fails, Python raises a `ValueError`; we catch it with `try/except`
and put that row in a "rejected" list instead of crashing.

**Why:** this is how a real data pipeline protects itself - one bad row must not stop the
whole load, and rejected rows must be saved so somebody can look at them later.

**Important idea:** a *type check* only asks "is this a number?". It does **not** ask
"is this number sensible?". An age of `150` or `-4` is a perfectly good integer, so it
passes here. We deal with those in Step 1.3 (a *validity check*).

In [3]:
import csv
import json
from collections import Counter

# Which columns must be numbers, and which Python type each one should become.
NUMERIC_COLUMNS = {
    "age": int,              # whole number of years
    "distance_km": float,    # can have decimals, e.g. 12.5
    "fee_inr": float,        # consultation fee in rupees
    "prior_no_shows": int,   # count of earlier no-shows
}


def parse_row(row):
    """Convert one raw CSV row (all text) into a row with proper number types.

    Raises ValueError if a number column is blank or is not a number.
    The error message starts with the column name so we know what went wrong.
    """
    clean = dict(row)  # copy, so the original row is left untouched
    for column, to_type in NUMERIC_COLUMNS.items():
        text = row[column].strip()
        try:
            clean[column] = to_type(text)      # int("") or float("abc") raises ValueError
        except ValueError as err:
            # Raise it again with the column name added, so the caller can record the reason.
            raise ValueError(f"{column}: {err}") from err
    return clean


accepted = []   # rows that passed the type check
rejected = []   # rows that failed, with the reason
rows_read = 0

# newline="" is what the csv module documentation recommends when opening CSV files.
with open(DATA / "appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)          # uses the header line as the dictionary keys
    # start=2 because line 1 of the file is the header
    for line_number, row in enumerate(reader, start=2):
        rows_read += 1
        try:
            accepted.append(parse_row(row))
        except ValueError as err:
            rejected.append({
                "line_number": line_number,
                "appointment_id": row["appointment_id"],
                "reason": str(err),
                "row": row,                 # keep the raw row exactly as it was
            })

# Save the rejected rows so they can be reviewed (never throw bad data away silently).
with open(OUT / "rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

print("Rows read    :", rows_read)
print("Rows accepted:", len(accepted))
print("Rows rejected:", len(rejected))

Rows read    : 1020
Rows accepted: 988
Rows rejected: 32


In [4]:
# Why were rows rejected? The reason text starts with the column name.
reasons = Counter(r["reason"].split(":")[0] for r in rejected)
print("Rejected by column:", dict(reasons))

print("\nExample rejected row:")
print(json.dumps(rejected[0], indent=2))

# A type check is not a validity check: impossible ages are still accepted here.
impossible = [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100]
print("\nImpossible ages that PASSED the type check:", impossible)
print("Types after parsing:", {c: type(accepted[0][c]).__name__ for c in NUMERIC_COLUMNS})

Rejected by column: {'distance_km': 20, 'age': 12}

Example rejected row:
{
  "line_number": 52,
  "appointment_id": "A0050",
  "reason": "distance_km: could not convert string to float: ''",
  "row": {
    "appointment_id": "A0050",
    "patient_id": "P117",
    "clinic_id": "C04",
    "booked_on": "2025-01-16",
    "appt_date": "2025-01-24",
    "age": "41",
    "gender": "M",
    "distance_km": "",
    "fee_inr": "600",
    "booking_channel": "Walk-in",
    "visit_type": "Consultation",
    "reminder_sent": "No",
    "prior_no_shows": "0",
    "status": "Attended"
  }
}

Impossible ages that PASSED the type check: [0, 150, -4, 250, 199]
Types after parsing: {'age': 'int', 'distance_km': 'float', 'fee_inr': 'float', 'prior_no_shows': 'int'}


**Reading the result**

- 1020 rows read, 988 accepted, 32 rejected. The 32 are 12 rows with a blank `age` and 20 rows
  with a blank `distance_km` (`int("")` and `float("")` both raise `ValueError`).
- The 5 impossible ages (0, 150, -4, 250, 199) were **accepted**, because each one is a valid
  integer. Catching them needs a business rule (age between 1 and 100), which comes in Step 1.3.
- Nothing is lost: the rejected rows sit in `outputs/rejected_rows.json`. In the pandas steps
  below we start again from the full raw file and *repair* these rows (fill the blanks)
  instead of dropping them.

## Step 1.2 - Load with pandas, remove duplicates, clean text, parse dates

**What:** load the same raw file with pandas, look at its shape and missing values, then
(1) drop exact duplicate rows, (2) clean the messy `booking_channel` spellings,
(3) turn the two date columns into real dates, and (4) drop rows where the appointment
date or the status is missing.

**Why:** duplicates would count the same appointment twice, messy spellings would split one
category into many, and we cannot use a row at all if we do not know *when* the appointment
was or *whether the patient came* (status is the thing we want to predict).

In [5]:
import pandas as pd

# Read the raw file. pandas guesses the column types and turns blanks (and text like "N/A") into NaN.
raw = pd.read_csv(DATA / "appointments.csv")

print("Shape (rows, columns):", raw.shape)
print("\nMissing values per column (only columns that have any):")
nulls = raw.isna().sum()
print(nulls[nulls > 0])
print("\nColumn types:")
print(raw.dtypes)

Shape (rows, columns): (1020, 14)

Missing values per column (only columns that have any):
appt_date       1
age            12
distance_km    20
status          4
dtype: int64

Column types:
appointment_id         str
patient_id             str
clinic_id              str
booked_on              str
appt_date              str
age                float64
gender                 str
distance_km        float64
fee_inr              int64
booking_channel        str
visit_type             str
reminder_sent          str
prior_no_shows       int64
status                 str
dtype: object


`age` shows as `float64` even though ages are whole numbers. That is because the column has
blanks, and the usual integer type in pandas cannot hold a missing value (`NaN` is a float).

### Duplicates
`duplicated()` marks a row as `True` when an earlier row is *exactly* the same in every column.

In [6]:
df = raw.copy()   # work on a copy; `raw` stays as loaded so we can always compare

n_dupes = df.duplicated().sum()
print("Exact duplicate rows:", n_dupes)

# Show one duplicated pair so we can see what a duplicate looks like.
example_id = df.loc[df.duplicated(), "appointment_id"].iloc[0]
display(df[df["appointment_id"] == example_id])

rows_before = len(df)
df = df.drop_duplicates()          # keeps the first copy, removes the repeats
print(f"Rows: {rows_before} -> {len(df)}")
print("appointment_id is now unique:", df["appointment_id"].is_unique)

Exact duplicate rows: 20


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended


Rows: 1020 -> 1000
appointment_id is now unique: True


### Clean `booking_channel`
There are only 3 real channels (App, Phone, Walk-in) but people typed them in different ways:
extra spaces, different capital letters. To a computer `"App"`, `"app"` and `" App"` are three
different values. We fix it by removing spaces (`str.strip`), lower-casing (`str.lower`) and then
mapping to one tidy label.

In [7]:
print("Different raw spellings:", df["booking_channel"].nunique())
print(sorted(df["booking_channel"].unique()))

tidy_labels = {"app": "App", "phone": "Phone", "walk-in": "Walk-in"}
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()        # remove spaces at the start and end
    .str.lower()        # make everything lower case
    .map(tidy_labels)   # replace with the tidy label (anything unknown would become NaN)
)

print("\nAfter cleaning:", df["booking_channel"].nunique(), "values")
print(df["booking_channel"].value_counts())
print("Unknown channels (should be 0):", df["booking_channel"].isna().sum())

Different raw spellings: 16
[' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']

After cleaning: 3 values
booking_channel
App        387
Phone      359
Walk-in    254
Name: count, dtype: int64
Unknown channels (should be 0): 0


### Parse the dates
`pd.to_datetime(..., errors="coerce")` turns text into real dates. `errors="coerce"` means:
if a value cannot be a date, do not crash - make it `NaT` ("Not a Time", the missing value for dates).

**Trap:** `isna()` earlier found only 1 missing `appt_date`. Parsing finds more, because text
such as `TBD` or an impossible date such as `2025-02-30` is *not blank* but is still *not a date*.

In [8]:
# Keep the original text for a moment so we can show which values failed.
appt_text = df["appt_date"]

df["booked_on"] = pd.to_datetime(df["booked_on"], format="%Y-%m-%d", errors="coerce")
df["appt_date"] = pd.to_datetime(df["appt_date"], format="%Y-%m-%d", errors="coerce")

print("booked_on that are not valid dates:", df["booked_on"].isna().sum())
print("appt_date that are not valid dates:", df["appt_date"].isna().sum())
print("\nThe appt_date values that failed:")
display(pd.DataFrame({
    "appointment_id": df.loc[df["appt_date"].isna(), "appointment_id"],
    "appt_date_as_typed": appt_text[df["appt_date"].isna()],
}))

booked_on that are not valid dates: 0
appt_date that are not valid dates: 6

The appt_date values that failed:


,appointment_id,appt_date_as_typed
98,A0094,not booked
167,A0162,2025-06-31
303,A0296,2025-04-31
504,A0494,TBD
534,A0523,NaN
678,A0666,2025-02-30


### Drop rows with a missing date or missing status
We do **not** guess these. A made-up appointment date would give a wrong lead time, and a
made-up status would teach the model wrong answers.

In [9]:
rows_before = len(df)
print("Missing booked_on:", df["booked_on"].isna().sum(),
      "| missing appt_date:", df["appt_date"].isna().sum(),
      "| missing status:", df["status"].isna().sum())

df = df.dropna(subset=["booked_on", "appt_date", "status"])
df = df.reset_index(drop=True)     # renumber the rows 0..n-1 after dropping

print(f"Rows: {rows_before} -> {len(df)}")
print("Row count so far: 1020 raw ->", 1020 - n_dupes, "after duplicates ->", len(df), "after missing date/status")
print("\nStatus counts:")
print(df["status"].value_counts())

Missing booked_on: 0 | missing appt_date: 6 | missing status: 4
Rows: 1000 -> 990
Row count so far: 1020 raw -> 1000 after duplicates -> 990 after missing date/status

Status counts:
status
Attended    772
No-Show     218
Name: count, dtype: int64


**Reading the result**

- Raw shape is (1020, 14). Missing: `age` 12, `distance_km` 20, `status` 4, `appt_date` 1.
- 20 exact duplicates removed: 1020 -> 1000 rows. Every `appointment_id` is now unique.
- 16 raw spellings of `booking_channel` became 3 clean values.
- 6 appointment dates were unusable (1 `N/A`, 2 pieces of text, 3 impossible calendar dates) and
  4 rows had no status. Dropping those 10 rows: 1000 -> 990.
- `age` and `distance_km` blanks are still there on purpose - we fill them in Step 1.3.

## Step 1.3 - Invalid ages, missing values and outliers

**What:** four things, and **the order matters**:

1. Turn impossible ages (below 1 or above 100) into missing values.
2. Fill missing `age` and `distance_km` with the **median** of the column.
3. Cap extreme `distance_km` and `fee_inr` values using the **IQR rule**.
4. Round the numbers to tidy values.

**Why this order:** if we computed the median *before* removing an age of 250, the bad value
would be part of the calculation. And if we capped outliers *before* filling blanks, the
quartiles would be worked out on a different set of rows. Clean first, then measure, then fill.

### 1) Invalid ages become missing
This is the *validity check* that the type check in Step 1.1 could not do.
`Series.mask(condition)` replaces the values where the condition is `True` with `NaN`.

In [10]:
import numpy as np

missing_age_before = df["age"].isna().sum()
invalid_age = (df["age"] < 1) | (df["age"] > 100)      # True for impossible ages

print("Ages that are blank already :", missing_age_before)
print("Ages that are impossible    :", invalid_age.sum(), "->", df.loc[invalid_age, "age"].tolist())

df["age"] = df["age"].mask(invalid_age)                # impossible -> NaN (missing)

print("Missing ages in total now   :", df["age"].isna().sum())

Ages that are blank already : 12
Ages that are impossible    : 5 -> [0.0, 150.0, -4.0, 250.0, 199.0]
Missing ages in total now   : 17


### 2) Fill missing values with the median
The **median** is the middle value when the numbers are sorted. We use it instead of the mean
because a few very large values (a patient who lives 60 km away) pull the mean up, but hardly
move the median. So the median is a safer "typical value".

In [11]:
age_median = df["age"].median()                # median ignores NaN by default
dist_median = df["distance_km"].median()
missing_dist = df["distance_km"].isna().sum()

print("Median age        :", age_median, "| rows to fill:", df["age"].isna().sum())
print("Median distance_km:", dist_median, "| rows to fill:", missing_dist)

df["age"] = df["age"].fillna(age_median)
df["distance_km"] = df["distance_km"].fillna(dist_median)

print("Missing after filling -> age:", df["age"].isna().sum(), "| distance_km:", df["distance_km"].isna().sum())

Median age        : 48.0 | rows to fill: 17
Median distance_km: 8.25 | rows to fill: 20
Missing after filling -> age: 0 | distance_km: 0


### 3) Cap outliers with the IQR rule
- **Q1** = the value below which 25% of the rows fall; **Q3** = the value below which 75% fall.
- **IQR** (inter-quartile range) = Q3 - Q1 = the width of the "middle half" of the data.
- **Upper fence** = Q3 + 1.5 x IQR. Anything above the fence is called an outlier.

We **cap** (also called *winsorise*): values above the fence are pulled down *to* the fence.
We do not delete the rows - a patient living very far away is real and is still "far"; we only
stop a handful of extreme numbers from dominating averages and the model.
`Series.clip(upper=...)` does the capping.

In [12]:
def iqr_cap(series, name):
    # Cap values above Q3 + 1.5*IQR and print what was done.
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    n_capped = (series > upper_fence).sum()
    print(f"{name:12s} Q1={q1}  Q3={q3}  IQR={iqr}  upper fence={upper_fence}  "
          f"capped={n_capped}  (max before={series.max()})")
    return series.clip(upper=upper_fence)


df["distance_km"] = iqr_cap(df["distance_km"], "distance_km")
df["fee_inr"] = iqr_cap(df["fee_inr"], "fee_inr")

print("\nMax after capping -> distance_km:", df["distance_km"].max(), "| fee_inr:", df["fee_inr"].max())

distance_km  Q1=4.625  Q3=13.0  IQR=8.375  upper fence=25.5625  capped=26  (max before=221.0)
fee_inr      Q1=500.0  Q3=787.5  IQR=287.5  upper fence=1218.75  capped=3  (max before=7500)

Max after capping -> distance_km: 25.5625 | fee_inr: 1218.75


### 4) Round
Age is a whole number, so we store it as an integer again (possible now that no value is
missing). Distance and fee are rounded to 2 decimals so the exported file is tidy.

In [13]:
df["age"] = df["age"].round().astype(int)
df["distance_km"] = df["distance_km"].round(2)
df["fee_inr"] = df["fee_inr"].round(2)

print("Missing values left in the whole table:", int(df.isna().sum().sum()))
display(df[["age", "distance_km", "fee_inr", "prior_no_shows"]].describe().round(2))

Missing values left in the whole table: 0


,age,distance_km,fee_inr,prior_no_shows
count,990.00,990.00,990.00,990.00
mean,48.17,9.50,635.71,0.64
std,16.66,6.16,183.80,0.95
min,18.00,0.60,300.00,0.00
25%,34.00,4.62,500.00,0.00
50%,48.00,8.25,650.00,0.00
75%,62.00,13.00,787.50,1.00
max,78.00,25.56,1218.75,5.00


**Reading the result**

- 5 impossible ages were set to missing. Together with the 12 that were already blank, that
  makes **17 missing ages**, all filled with the median age of **48**.
- Missing distances were filled with the median **8.25 km**.
- `distance_km`: Q1 4.625, Q3 13.0, upper fence 25.5625 -> **26** values capped.
- `fee_inr`: Q1 500, Q3 787.5, upper fence 1218.75 -> **3** values capped.
- The table now has 990 rows and no missing values.

## Step 1.4 - New columns and NumPy calculations

**What:** create two new columns, then do a few calculations with NumPy arrays only (no loops).

- `lead_days` = how many days the patient had to wait between booking and the appointment.
- `no_show` = 1 if the patient did not come, 0 if they attended. This is our **target** for Part 3.

**Why:** models and SQL need numbers, not text like "No-Show". And lead time is a classic
driver of no-shows: the longer the wait, the easier it is to forget or change plans.

In [14]:
# Subtracting two date columns gives a "timedelta"; .dt.days turns it into a whole number of days.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Comparison gives True/False; astype(int) turns that into 1/0.
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("no-shows:", df["no_show"].sum(), "of", len(df), f"= {df['no_show'].mean():.1%}")

lead_days range: 1 to 45
no-shows: 218 of 990 = 22.0%


### NumPy only, no loops
A NumPy array is a block of numbers of one type. Operations apply to the whole array at once
(this is called **vectorisation**) - shorter to write and much faster than a Python `for` loop.

- `axis=0` means "go down the rows", so `mean(axis=0)` gives one mean **per column**.
- A **boolean mask** is an array of True/False. `mask.sum()` counts the Trues, `mask.mean()` gives the share.
- A **z-score** says how many standard deviations a value is from the mean: `z = (x - mean) / std`.
  `|z| > 2` is a common rule of thumb for "unusually far from average".

In [15]:
numeric_columns = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[numeric_columns].to_numpy(dtype=float)     # 2-D array: 990 rows x 4 columns
print("Array shape:", X.shape)

# 1) Mean of every column in one call.
column_means = X.mean(axis=0)
print("Column means", numeric_columns, "=", np.round(column_means, 2))

# 2) Boolean mask: appointments booked more than 14 days ahead.
lead = X[:, 3]                       # all rows, column 3 (lead_days)
long_lead = lead > 14                # array of True/False
print(f"lead_days > 14: {long_lead.sum()} rows ({long_lead.mean():.1%})")

# 3) z-score of distance, then count the unusual ones.
dist = X[:, 1]
z = (dist - dist.mean()) / dist.std()      # one z value per row, no loop
unusual = np.abs(z) > 2
print(f"|z| > 2 on distance_km: {unusual.sum()} rows")
print(f"   (distance mean={dist.mean():.2f}, std={dist.std():.2f}, "
      f"so |z|>2 means further than {dist.mean() + 2 * dist.std():.2f} km)")

# Bonus check with the same mask idea: is the no-show rate different for long lead times?
y = df["no_show"].to_numpy()
print(f"No-show rate when lead_days > 14 : {y[long_lead].mean():.1%}")
print(f"No-show rate when lead_days <= 14: {y[~long_lead].mean():.1%}")

Array shape: (990, 4)
Column means ['age', 'distance_km', 'fee_inr', 'lead_days'] = [ 48.17   9.5  635.71  11.82]
lead_days > 14: 296 rows (29.9%)
|z| > 2 on distance_km: 53 rows
   (distance mean=9.50, std=6.15, so |z|>2 means further than 21.81 km)
No-show rate when lead_days > 14 : 34.5%
No-show rate when lead_days <= 14: 16.7%


**Reading the result**

- `lead_days` runs from 1 to 45 days. 218 of 990 appointments are no-shows = **22.0%**.
- Column means are about `[48.17, 9.50, 635.71, 11.82]`: the typical patient is 48 years old,
  lives 9.5 km away, pays about Rs 636 and waits about 12 days.
- 296 appointments (29.9%) are booked more than 14 days ahead.
- 53 rows have a distance more than 2 standard deviations from the mean. All of them are on
  the *far* side, because distance cannot be negative enough to be 2 std below the mean.
- The last two lines already hint at the main story of this project: long waits go together
  with many more no-shows. We will measure this properly in Part 2 and Part 3.